# MiniCells Core Validation 002 — Write Addressability under Superposition

Formal run for the frozen `core-validation-002` protocol. The editor never receives the ground-truth sparse code or target feature identity. The decisive comparison holds the learned sparse representation fixed and changes only the write operator.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json

BRANCH = 'codex/core-validation-002-write-addressability'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','checkout',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('HEAD', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('TREE', subprocess.check_output(['git','rev-parse','HEAD^{tree}'], text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/research/04-continual-learning-core/test_core_validation_002.py'], check=True)
subprocess.run([sys.executable,'scripts/research/run_core_validation_002.py','--smoke','--device','cpu'], check=True)
print('Core Validation 002 tests and CPU smoke passed.')


In [ ]:
import torch
print({'torch':torch.__version__,'cuda':torch.version.cuda,'gpu_count':torch.cuda.device_count(),'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.is_available(), 'Formal Core Validation 002 requires a CUDA GPU.'


In [ ]:
OUT = ROOT / 'results' / 'core-validation-002-write-addressability'
if OUT.exists():
    import shutil
    shutil.rmtree(OUT)
subprocess.run([sys.executable,'scripts/research/run_core_validation_002.py','--device','cuda'], check=True)
subprocess.run([sys.executable,'scripts/research/report_core_validation_002.py'], check=True)


In [ ]:
decision=json.loads((OUT/'decision.json').read_text())
print(json.dumps(decision, indent=2, sort_keys=True))
import pandas as pd
display(pd.read_csv(OUT/'seed-summary.csv'))


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT/'update-error-vs-write-leakage.png')))
display(Image(filename=str(OUT/'sequential-write-leakage.png')))
display(Image(filename=str(OUT/'mechanistic-leakage-prediction.png')))


## Optional recovery-load sweep

Run this only after the frozen primary result. The sweep is descriptive in v1 and does not change the formal decision.


In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    subprocess.run([sys.executable,'scripts/research/run_core_validation_002.py','--device','cuda','--sweep'], check=True)
    subprocess.run([sys.executable,'scripts/research/report_core_validation_002.py'], check=True)
    display(Image(filename=str(OUT/'recovery-load-sweep.png')))


## Publish

The final cell publishes curated formal results to `kaggle/core-validation-002-write-addressability-results`. It expects the existing Kaggle secret `GITHUB_TOKEN`.


In [ ]:
subprocess.run([sys.executable,'scripts/research/publish_core_validation_002.py','--push'], check=True)
print('Published Core Validation 002 results branch.')
